# Sistema de Recomendação B2C (Cross-Sell)

**Objetivo de Negócio:** Como descobrimos que a nossa receita depende de uma enorme massa de consumidores do varejo (B2C), a melhor forma de aumentar a receita sem adquirir novos clientes é aumentar o ticket médio e a taxa de recompra. 

Neste notebook, construiremos um **Motor de Recomendação** transparente e objetivo (sem redes neurais complexas), baseado na afinidade de produtos (o clássico "Quem comprou X, também levou Y").

In [ ]:
import duckdb
import pandas as pd

# Conectar ao DuckDB
con = duckdb.connect()

# Usaremos a tabela Fato de Vendas que já contém os order_items tratados
fato_vendas = r"E:\repo\lh_nautical_analise\data\processed\fato_vendas.parquet"

print("Bases carregadas e prontas para processamento.")

## 1. Estratégia Adotada: Filtro Colaborativo Baseado em Itens (Co-ocorrência)

Para evitar modelos de *machine learning* opacos (Black Box), utilizaremos a força analítica do **SQL (DuckDB)** para criar uma Matriz de Co-ocorrência.

**Lógica:**
Se o Carrinho 1 contém o 'Caiaque A' e o 'Remo B', nós registramos +1 na afinidade entre eles. Fazendo isso para centenas de milhares de pedidos, revelamos os padrões ocultos de consumo cruzado.

In [ ]:
# Construindo as afinidades no banco de dados
query_afinidade = f"""
WITH itens_carrinho AS (
    SELECT 
        order_id,
        sku,
        product_name
    FROM read_parquet('{fato_vendas}')
),
pares AS (
    SELECT 
        a.sku AS id_produto_ancora,
        a.product_name AS nome_produto_ancora,
        b.sku AS id_produto_recomendado,
        b.product_name AS nome_produto_recomendado,
    FROM itens_carrinho a
    JOIN itens_carrinho b 
        ON a.order_id = b.order_id 
        AND a.sku != b.sku
)
SELECT 
    id_produto_ancora,
    nome_produto_ancora,
    id_produto_recomendado,
    nome_produto_recomendado,
    COUNT(*) as vezes_comprados_juntos
FROM pares
GROUP BY 1, 2, 3, 4
HAVING vezes_comprados_juntos > 5 -- Filtro de ruído (mínimo de relevância)
ORDER BY vezes_comprados_juntos DESC
"""

# Executar e guardar na memória
df_afinidade = con.execute(query_afinidade).df()

print(f"Foram encontradas {len(df_afinidade):,} regras de associação fortes na base de vendas.")
df_afinidade.head(10)

## 2. Simulador de Recomendação (Uso em Produção)

Vamos criar a função exata que seria embarcada no e-commerce da LH Nautical. Quando o usuário entrar na página de um produto, o sistema busca os 3 itens com maior grau de co-ocorrência.

In [ ]:
def recomendar_produtos(produto_id, top_n=3):
    # Filtrar o dataframe pré-calculado para o produto âncora
    recomendacoes = df_afinidade[df_afinidade['id_produto_ancora'] == produto_id]
    
    if recomendacoes.empty:
        return "Sem dados suficientes para recomendar (Tentar os Mais Vendidos da Categoria)."
    
    # Pegar os Top N produtos mais comprados junto
    top_recs = recomendacoes.head(top_n)
    
    nome_ancora = top_recs.iloc[0]['nome_produto_ancora']
    
    print(f"🛍️ Clientes que compraram ['{nome_ancora}'] também levaram:")
    for i, row in top_recs.iterrows():
        print(f"  -> {row['nome_produto_recomendado']} (Comprados juntos {row['vezes_comprados_juntos']} vezes)")
        
# Simulando o uso na loja para o produto de SKU aleatório (Exemplo)
if not df_afinidade.empty:
    # Pegar um ID aleatório dos mais populares para teste
    produto_teste = df_afinidade['id_produto_ancora'].iloc[0]
    recomendar_produtos(produto_teste)

## 3. Insight de Negócio e Entrega (F-H-R)

**Fato Observado:** A aplicação da Matriz de Co-ocorrência via SQL foi capaz de extrair milhares de associações diretas entre produtos, baseadas unicamente no histórico real de carrinhos de compra, de forma extremamente rápida e computacionalmente barata.

**Hipótese:** Modelos simples e transparentes de *Market Basket Analysis* (Análise de Cesta) funcionam muito melhor para o estágio atual da LH Nautical do que sistemas baseados em Deep Learning, pois a diretoria compreende a lógica, confia no número de vendas conjuntas e consegue auditar as sugestões.

**Recomendação:** 
- **Tecnologia:** Exportar essa tabela `df_afinidade` como um banco NoSQL (ex: Redis) e acoplar na página de detalhe de produto (PDP) e no carrinho do E-commerce sob a tag *"Você também pode gostar"*.
- **Marketing:** A equipe pode usar essa tabela para disparar e-mails: se o cliente comprou o produto A ontem, envie um cupom do produto B amanhã.
- **Limitações a observar:** O modelo não recomenda lançamentos (Cold Start). Para produtos novos sem histórico de vendas, o e-commerce deverá configurar recomendações alternativas (ex: "Lançamentos da mesma Categoria").